# GeoParquet Persistence

Each isochrone result can be saved to disk and reloaded with full fidelity —
including the original request, all GeoDataFrames, and the router name. This
makes long batch runs resumable and results shareable without re-running routing.

**What's covered**
1. Writing a result with `write_result()` / `result.to_parquet()`
2. Reading it back with `read_result()` / `TravelTimeResult.from_parquet()`
3. The on-disk layout and what each file contains
4. `result_id()` — the deterministic directory naming scheme
5. Idempotency: re-running the same request overwrites, not duplicates
6. Inspecting the GeoParquet files directly with geopandas
7. Building a simple result store and listing what's in it


## Setup

In [ ]:
import tempfile
from pathlib import Path

from idd_travel_time.constants import TransportMode
from idd_travel_time.io import read_result, result_id, write_result
from idd_travel_time.isochrone import compute
from idd_travel_time.models import TravelTimeRequest, TravelTimeResult

# Use a temp dir for this notebook — change to a real path for real work
RESULTS_DIR = Path(tempfile.mkdtemp()) / "isochrone_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results will be written to: {RESULTS_DIR}")

## 1 · Compute and save

In [ ]:
request = TravelTimeRequest.from_place(
    "Kenyatta National Hospital, Nairobi",
    TransportMode.WALKING,
    duration_minutes=20,
)

result = compute(request)
print(f"Computed isochrone via: {result.router_name}")

In [ ]:
# Save — returns the result directory path
result_dir = write_result(result, RESULTS_DIR)
print(f"Saved to: {result_dir}")
print("\nFiles on disk:")
for f in sorted(result_dir.iterdir()):
    print(f"  {f.name:25}  {f.stat().st_size:>8} bytes")

## 2 · The result directory layout

```
{result_id}/
├── meta.json          request provenance + schema version
├── isochrone.parquet  always present — the polygon
├── network.parquet    road edges (present when router produced them)
└── poi.parquet        named places (present when fetched)
```


In [ ]:
import json

meta = json.loads((result_dir / "meta.json").read_text())
print(json.dumps(meta, indent=2))

## 3 · Read back

In [ ]:
loaded = read_result(result_dir)

print(f"Router:          {loaded.router_name}")
print(
    f"Origin:          ({loaded.request.origin_lat:.6f}, {loaded.request.origin_lon:.6f})"
)
print(f"Mode:            {loaded.request.mode.value}")
print(f"Duration:        {loaded.request.duration_minutes} min")
print(f"Isochrone rows:  {len(loaded.isochrone_gdf)}")
print(
    f"Network:         {len(loaded.network_gdf) if loaded.network_gdf is not None else 'None'}"
)
print(
    f"POIs:            {len(loaded.poi_gdf) if loaded.poi_gdf is not None else 'None'}"
)

## 4 · Convenience methods on TravelTimeResult

In [ ]:
# Equivalent to write_result(result, RESULTS_DIR)
result_dir2 = result.to_parquet(RESULTS_DIR)

# Equivalent to read_result(result_dir)
loaded2 = TravelTimeResult.from_parquet(result_dir2)
print(f"Round-trip successful: {loaded2.request.mode.value}")

## 5 · `result_id()` — deterministic naming

The directory name is derived entirely from the request parameters.
The same request always maps to the same name — re-running overwrites,
never duplicates.


In [ ]:
rid = result_id(request)
print(f"Result ID: {rid}")
print()

# Two identical requests → same id
request2 = TravelTimeRequest.from_place(
    "Kenyatta National Hospital, Nairobi",
    TransportMode.WALKING,
    duration_minutes=20,
)
print(f"Same request → same id: {result_id(request) == result_id(request2)}")

# Different duration → different id
request3 = TravelTimeRequest.from_place(
    "Kenyatta National Hospital, Nairobi",
    TransportMode.WALKING,
    duration_minutes=30,
)
print(f"Different duration → different id: {result_id(request) != result_id(request3)}")

## 6 · Idempotency — re-running overwrites

In [ ]:
# Run write twice — still only one directory in the store
write_result(result, RESULTS_DIR)
write_result(result, RESULTS_DIR)

subdirs = [p for p in RESULTS_DIR.iterdir() if p.is_dir()]
print(f"Number of result directories: {len(subdirs)}")
print("(Re-runs overwrite, not accumulate)")

## 7 · Inspecting GeoParquet files directly

In [ ]:
import geopandas as gpd

# Read the parquet files with geopandas directly — useful for ad-hoc analysis
iso = gpd.read_parquet(result_dir / "isochrone.parquet")
print("Isochrone GeoDataFrame:")
print(iso)
print(f"\nCRS: {iso.crs}")
print(f"Geometry type: {iso.geometry.geom_type.iloc[0]}")

# Reproject to UTM for area calculation
iso_utm = iso.to_crs(iso.estimate_utm_crs())
area_km2 = iso_utm.geometry.area.iloc[0] / 1e6
print(f"\nReachable area: {area_km2:.2f} km²")

## 8 · Building a simple result store

In [ ]:
# Compute a few results and build a summary table
import pandas as pd

locations = [
    ("Kenyatta National Hospital, Nairobi", "KE", 20),
    ("Mulago National Referral Hospital, Kampala", "UG", 20),
]

rows = []
for place, hint, dur in locations:
    req = TravelTimeRequest.from_place(
        place, TransportMode.WALKING, country_hint=hint, duration_minutes=dur
    )
    res = compute(req)
    write_result(res, RESULTS_DIR)

    iso_utm = res.isochrone_gdf.to_crs(res.isochrone_gdf.estimate_utm_crs())
    rows.append(
        {
            "place": place.split(",")[0],
            "lat": round(req.origin_lat, 4),
            "lon": round(req.origin_lon, 4),
            "duration_min": req.duration_minutes,
            "area_km2": round(iso_utm.geometry.area.iloc[0] / 1e6, 3),
            "result_id": result_id(req),
        }
    )

summary = pd.DataFrame(rows)
summary

In [ ]:
# Reload every result from disk and verify
print("Loading all results from disk:")
for d in sorted(RESULTS_DIR.iterdir()):
    if d.is_dir():
        r = read_result(d)
        print(
            f"  ✓  {r.request.origin_lat:.4f}, {r.request.origin_lon:.4f}  "
            f"{r.request.mode.value} {r.request.duration_minutes}min  "
            f"router={r.router_name}"
        )